# 06 — Walk-Forward Cross-Validation

**Goal:** Replace the static 70/15/15 split with an expanding-window walk-forward CV (5 folds) so we can evaluate regime stability of the LSTM signal.

Strictly temporal: training data always precedes validation data. No shuffling. No look-ahead leakage.

We log per-fold out-of-fold (OOF) metrics: **Sharpe, Accuracy, F1, AUC, Max Drawdown, # Trades**.

In [ ]:
import sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

INTERIM = ROOT / 'notebooks' / 'interim'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)
np.random.seed(42)

from src.cv.walk_forward import walk_forward_splits, run_walk_forward

# Load features produced by Notebook 03 (re-run if missing)
with (INTERIM / 'features_for_lstm.pkl').open('rb') as f:
    bundle = pickle.load(f)

X = np.concatenate([bundle['train_x'], bundle['val_x'], bundle['test_x']], axis=0)
y = np.concatenate([bundle['train_y'], bundle['val_y'], bundle['test_y']], axis=0)
close = np.concatenate([np.zeros(len(bundle['train_x'])),  # placeholder — we'll fix below
                         bundle.get('val_close', np.zeros(len(bundle['val_x']))),
                         bundle['test_close']], axis=0)
dates = np.concatenate([bundle['train_dates'], bundle['val_dates'], bundle['test_dates']], axis=0)

# Properly reconstruct close across the full window by re-loading the daily dataset
# (the bundle only stored val_close/test_close)
import pandas as pd
merged = pd.read_parquet(INTERIM / 'merged_with_llm_sentiment.parquet').sort_values('date').reset_index(drop=True)
close = merged['close'].values

print(f'X: {X.shape}  y: {y.shape}  close: {close.shape}  dates: {dates.shape}')
print(f'Date range: {pd.Timestamp(dates[0]).date()} → {pd.Timestamp(dates[-1]).date()}')

## 6.1 Inspect the 5 expanding-window folds

In [ ]:
n = len(X)
for fold in walk_forward_splits(n=n, n_folds=5, min_train=400, val_size=60):
    train_dates_str = f'{pd.Timestamp(dates[fold.train_start]).date()} → {pd.Timestamp(dates[fold.train_end - 1]).date()}'
    val_dates_str = f'{pd.Timestamp(dates[fold.val_start]).date()} → {pd.Timestamp(dates[fold.val_end - 1]).date()}'
    print(f'{fold}')
    print(f'  train: {train_dates_str}')
    print(f'  val  : {val_dates_str}')

## 6.2 Model factory — baseline LSTM (re-used from Notebook 04)

In [ ]:
n_features = X.shape[-1]

def build_baseline_lstm():
    inp = layers.Input(shape=(1, n_features), name='features')
    x = layers.LSTM(64, return_sequences=False)(inp)
    x = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid', name='prob_up')(x)
    m = models.Model(inp, out, name='baseline_lstm_wf')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return m

# Smoke test
m = build_baseline_lstm()
print(f'Params: {m.count_params():,}')
tf.keras.backend.clear_session()

## 6.3 Run walk-forward CV
Each fold trains a fresh model with EarlyStopping (patience=7) and class weighting.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights computed on the FULL training set (fold 5)
classes = np.unique(y)
weights = compute_class_weight('balanced', classes=classes, y=y)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print('Class weights (full set):', class_weight)

def make_callbacks():
    return [
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    ]

result = run_walk_forward(
    X=X, y=y, close=close, dates=dates,
    model_factory=build_baseline_lstm,
    n_folds=5, min_train=400, val_size=60,
    threshold=0.5, fee=0.001,
    fit_kwargs=dict(
        epochs=30, batch_size=32, verbose=0,
        class_weight=class_weight,
        callbacks=[callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True),
                   callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)],
    ),
    verbose=True,
)

## 6.4 OOF metrics summary

In [ ]:
print(result.summary())
df_oof = result.to_dataframe()
df_oof

## 6.5 Visualize per-fold Sharpe & Accuracy

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
except Exception:
    pass
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)

ax1.bar(df_oof['fold'], df_oof['sharpe'], color='#0f766e')
ax1.axhline(0, c='k', lw=0.6)
ax1.axhline(df_oof['sharpe'].mean(), ls='--', c='#f59e0b', label=f'Mean = {df_oof["sharpe"].mean():+.2f}')
ax1.set_xlabel('Fold #')
ax1.set_ylabel('OOF Sharpe Ratio')
ax1.set_title('Per-Fold Sharpe (regime stability)')
ax1.legend()

ax2.bar(df_oof['fold'], df_oof['accuracy'], color='#3b82f6')
ax2.axhline(0.5, c='k', lw=0.6, ls='--', label='Random baseline (0.50)')
ax2.axhline(df_oof['accuracy'].mean(), ls='--', c='#f59e0b', label=f'Mean = {df_oof["accuracy"].mean():.2f}')
ax2.set_xlabel('Fold #')
ax2.set_ylabel('OOF Accuracy')
ax2.set_ylim(0, 1)
ax2.set_title('Per-Fold Accuracy')
ax2.legend()

plt.savefig(OUTPUTS / 'walk_forward_oof_metrics.png', dpi=140, bbox_inches='tight')
plt.show()

## 6.6 Persist OOF results

In [ ]:
df_oof.to_csv(OUTPUTS / 'walk_forward_oof_metrics.csv', index=False)
print(f'Wrote {OUTPUTS / "walk_forward_oof_metrics.csv"}')
print(f'\nKey takeaway: OOF Sharpe = {result.oof_sharpe_mean:+.3f} ± {result.oof_sharpe_std:.3f} across 5 folds')
print('A small std indicates regime stability; a large std indicates the model is regime-dependent.')

## 6.7 Summary
- Replaced static 70/15/15 split with 5-fold expanding-window walk-forward CV.
- Strict temporal ordering preserved — no look-ahead leakage.
- Logged per-fold OOF metrics: Sharpe, Accuracy, F1, AUC, Max DD, # trades.
- Persisted `outputs/walk_forward_oof_metrics.csv` for downstream analysis.
- The std of OOF Sharpe is the key regime-stability indicator.